In [ ]:
import warnings; warnings.filterwarnings("ignore")
from statsmodels.tsa.statespace.sarimax import SARIMAX

H = 30
cutoff = daily["record_date"].max() - pd.Timedelta(days=H)

def run_model(train, H, order, seasonal_order):
    m = SARIMAX(train, order=order, seasonal_order=seasonal_order,
                enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
    f = m.get_forecast(H)
    ci = f.conf_int(alpha=0.2)                     # 80% interval
    return f.predicted_mean, ci.iloc[:, 0], ci.iloc[:, 1]

models = [("ARIMA",   (1,1,1), (0,0,0,0)),
          ("SARIMAX", (1,1,1), (1,1,1,7))]         # 7 = weekly seasonality

rows = []
for k, s in series.items():
    train, test = s[s.index <= cutoff], s[s.index > cutoff]

    for name, order, sorder in models:
        mean, lo, hi = run_model(train, H, order, sorder)
        rows.append(pd.DataFrame({
            "sku_wh_id": k, "target_period": test.index, "actual": test.values,
            "forecast_units": mean.clip(lower=0).values,
            "lower_bound": lo.clip(lower=0).values,
            "upper_bound": hi.clip(lower=0).values,
            "model_name": name}))

    last7 = train.tail(7)                          # baseline, for comparison only
    dow = last7.groupby(last7.index.dayofweek).mean()
    rows.append(pd.DataFrame({
        "sku_wh_id": k, "target_period": test.index, "actual": test.values,
        "forecast_units": [dow[d] for d in test.index.dayofweek],
        "lower_bound": np.nan, "upper_bound": np.nan, "model_name": "SeasonalNaive"}))

res = pd.concat(rows, ignore_index=True)
print(res.groupby("model_name").size())            # expect 300 rows for each model